In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from tqdm import tqdm

ROOT_DIR = "/media/ravi-bullock/Elements/VoxelWiseSaliency_Roberto/Roberto_ERF_Local_Adult/Subject_Wise_Saliency"
OUT_CSV = "Adult_VOX1_Final_Metrics.csv"

target_rois = {
    "Left_Lateral_Ventricle": [92],
    "Right_Lateral_Ventricle": [41],
    "Left_Thalamus": [91],
    "Right_Thalamus": [40]
}


def calculate_occupancy(grad_map, roi_mask, brain_mask):
    """Ratio of TOTAL saliency mass: ROI vs Whole Brain."""
    total_brain_sal = np.sum(grad_map[brain_mask > 0])
    if total_brain_sal <= 0: return 0
    sal_in_roi = np.sum(grad_map[roi_mask > 0])
    return sal_in_roi / total_brain_sal

def calculate_rsm(grad_map, roi_mask, brain_mask):
    """Ratio of MEAN saliency intensity: ROI vs Non-ROI Brain Tissue."""
    inside_pts = grad_map[roi_mask > 0]
    outside_pts = grad_map[(brain_mask > 0) & (roi_mask == 0)]
    
    inside_mean = np.mean(inside_pts) if inside_pts.size > 0 else 0
    outside_mean = np.mean(outside_pts) if outside_pts.size > 0 else 1e-8
    
    return inside_mean / (outside_mean + 1e-12)

results = []
subjects = sorted([d for d in os.listdir(ROOT_DIR) if d.startswith("sub-")])

for sub in tqdm(subjects, desc="Analyzing Adult VOX1"):
    sub_path = os.path.join(ROOT_DIR, sub)
    
    mask_path = os.path.join(sub_path, f"{sub}_binary_mask.nii.gz")
    
    if not os.path.exists(mask_path):
        continue
        
    try:
        brain_mask_nii = nib.load(mask_path)
        brain_mask = brain_mask_nii.get_fdata()

        for roi_name, labels in target_rois.items():
            roi_folder = os.path.join(sub_path, roi_name)
            sal_path = os.path.join(roi_folder, f"{sub}_{roi_name}_saliency.nii.gz")
            seg_path = os.path.join(roi_folder, f"{sub}_CerebrA_native.nii.gz")
            if os.path.exists(sal_path) and os.path.exists(seg_path):
                grad_map = nib.load(sal_path).get_fdata()
                seg_data = nib.load(seg_path).get_fdata()
                
                roi_mask = np.isin(seg_data, labels)
                
                occ = calculate_occupancy(grad_map, roi_mask, brain_mask)
                rsm = calculate_rsm(grad_map, roi_mask, brain_mask)
                
                results.append({
                    "Subject": sub,
                    "ROI": roi_name,
                    "Fractional_Occupancy": occ,
                    "RSM": rsm
                })
    except Exception as e:
        print(f"⚠️ Skipping {sub} due to error: {e}")

if not results:
    print("❌ No results collected. Check your file paths.")
else:
    df = pd.DataFrame(results)
    
    summary = df.groupby("ROI")[["Fractional_Occupancy", "RSM"]].agg(['mean', 'std'])
    
    print("\n--- Final Adult VOX1 Saliency Metrics ---")
    print(summary)
    
    df.to_csv(OUT_CSV, index=False)
    print(f"\n✅ Results saved to {OUT_CSV}")